# ERA5 from the NCAR S3 bucket — quick start

The analysis modules in `ERA5/surface_energy_budget` read their data through one function,
`seb_analysis_common.load_seb_data`. With `storage="aws"` that function returns a lazy dataset
over `s3://nsf-ncar-era5` instead of opening `data/<region>/*.nc`; everything downstream —
`prepare()`, the streaming passes, the figures — is the same code. This notebook shows the
three ways in: the raw reader, a `prepare()` call, and the batch driver.

Nothing here needs an AWS account: the bucket is public and read anonymously. An account is
only needed to run big jobs *inside* us-west-2 (see `remote/` and the README).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise FileNotFoundError("no .git above " + str(start))

SEB_DIR = repo_root(Path.cwd().resolve()) / "ERA5" / "surface_energy_budget"
if str(SEB_DIR) not in sys.path:
    sys.path.insert(0, str(SEB_DIR))
warnings.filterwarnings("ignore", category=FutureWarning)

from aws_pipeline import era5_s3, s3_storage
print("cache:", era5_s3.CACHE_DIR, "| workers:", era5_s3.n_workers())

## 1. The raw reader

`open_dataset(N, W, S, E, windows, variables)` builds the lazy dataset from S3 listings alone
(~1 s, no data read). Loading a slice fetches exactly the HDF5 chunks that slice touches, in
parallel, and caches the decoded window on disk — a second `.load()` of the same hours is
instant.

In [ ]:
# Barrow strip, first four days of October 2024, five variables
ds = era5_s3.open_dataset(80, -165, 70, -150, [("2024-10-01", "2024-10-04")],
                          variables=["tcc", "tclw", "tciw", "siconc", "msdwlwrf", "tp"],
                          month_align=False)
ds

In [ ]:
import time
t0 = time.time()
day = ds.isel(valid_time=slice(0, 24)).load()       # 24 h x 6 variables
print(f"loaded in {time.time() - t0:.1f} s  (again from cache: ", end="")
t0 = time.time(); ds.isel(valid_time=slice(0, 24)).load(); print(f"{time.time() - t0:.2f} s)")

fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
for ax, (name, kw) in zip(axs, [("tclw", dict(vmin=0, vmax=0.3, cmap="Blues")),
                                ("msdwlwrf", dict(vmin=180, vmax=320, cmap="inferno")),
                                ("siconc", dict(vmin=0, vmax=1, cmap="Blues_r"))]):
    m = ax.pcolormesh(day.longitude, day.latitude, day[name].isel(valid_time=12), shading="nearest", **kw)
    ax.set_title(f"{name} [{day[name].attrs['units']}]  {str(day.valid_time.values[12])[:13]}")
    fig.colorbar(m, ax=ax, shrink=0.85)
plt.show()

### What a bigger request would move

The cost is set by the HDF5 chunking of each group, not by the box: flux and pressure-level
chunks are whole-globe slabs. `describe_cost` says so before you commit.

In [ ]:
wins = era5_s3.season_windows(range(2014, 2025), (10, 1), (3, 31))
print("all 33 SEB variables, 11 cold seasons:")
print(era5_s3.describe_cost(wins))
print()
print("only what figures 1-3 and 5-6 read (tcc tclw tciw siconc tp):")
print(era5_s3.describe_cost(wins, ["tcc", "tclw", "tciw", "siconc", "tp"]))

## 2. The analysis modules, unchanged

`region` is any name in `era5_seb_variables.REGIONS`, a box registered with
`s3_storage.register_region`, or an inline `"box:N,W,S,E"`. The time window is taken from
`years` + `season_start/season_end` exactly as the season scripts define it. This is the Ocean
Visions notebook's `COMMON` dict with `storage="aws"`; one season here to keep it short (a few
minutes from a laptop the first time, seconds afterwards).

In [ ]:
import plot_lwp_histogram_by_surface_class as lwph

COMMON = dict(
    region="barrow",
    years=(2024,),                     # season START years; the talk used range(2014, 2025)
    season_start=(10, 1), season_end=(3, 31),
    phase_mode="fraction", liquid_fraction_min=0.90, min_lwp=0.01, min_iwp=0.01,
    min_cloud_fraction=0.95, lsm_tol=0.01,
    storage="aws",                     # <- the only change from the local run
    dpi=110,
)
PRECIP = dict(no_precip=True, precip_var="rate", precip_rate_max=0.05)

A = lwph.prepare(**COMMON, **PRECIP, ice_fraction_min=0.90)
lwph.print_report(A)

In [ ]:
lwph.fig_era5_vs_obs_simple_forOV(A, out_dir=None, halo_alpha=0.7);

### A different box

Register it once per session (or use `region="box:82,-175,65,-110"` inline). The adapter warns
when the box does not contain the ARM Utqiaġvik cell, because figures 1–3 and 7a compare that
cell with the ARM record and would silently use an edge cell instead.

In [ ]:
import map_liquid_hours as mlh

s3_storage.register_region("beaufort_wide", 82, -175, 65, -110, "Beaufort-Chukchi, wide")
win = era5_s3.make_window(82, -175, 65, -110)
print(f"{win.n_lat} x {win.n_lon} = {win.n_lat * win.n_lon:,} cells "
      f"({win.n_lat * win.n_lon / 2501:.0f}x Barrow); ARM cell inside:",
      s3_storage.site_in_domain(era5_s3.open_dataset(82, -175, 65, -110, [('2024-10-01', '2024-10-01')],
                                                     variables=['tcc'], verbose=False)))

# Uncomment to run: ~1 GB of analysis tiles + 6.5 GB of mtpr for one season from a laptop.
# M = mlh.prepare_maps(**{**COMMON, "region": "beaufort_wide"}, **PRECIP, ice_fraction_min=0.85)
# mlh.fig_fraction_and_lwp(M, out_dir=None);

## 3. The batch driver

`run_analysis.py` runs the figure set for any box/season/storage and saves figures plus
pickles of the reduced results. The invocation is identical on the laptop and on an EC2
instance in us-west-2 (`remote/run_job.sh` wraps it there).

In [ ]:
!python run_analysis.py --help | head -40

## 4. Verification against the local archive

`verify_against_local.py variables` compares every variable cell by cell with the CDS files in
`data/<region>/`; `analysis` runs `lwph.prepare` both ways for a season and compares `A.col`.

In [ ]:
from aws_pipeline import verify_against_local
verify_against_local.main(["variables", "--region", "barrow", "--start", "2024-10-01", "--end", "2024-10-02"]);